# Lab 1 · Del PDF a la estructura

**Sesión Norma+ · Diplomado de Ciencia de Datos e IA para Ciencias Sociales**

Antes de pedirle nada a un LLM, hay que saber *dónde* está lo que buscamos. En este lab convertimos tres resoluciones en texto limpio, identificamos el tipo de procedimiento, construimos un mapa de secciones y aislamos la **zona argumentativa**, que es donde viven los criterios jurídicos.

Este lab **no usa API**: todo es determinístico, con reglas y expresiones regulares.

| Paso | Qué hacemos | Qué aprendemos |
|---|---|---|
| 1 | Leer el PDF página por página | Trabajar con texto *y* posición |
| 2 | Detectar y quitar el pie repetido | Ruido estructural |
| 3 | Clasificar el procedimiento | Un router con perfiles |
| 4 | Mapear secciones y apartados | Offsets exactos |
| 5 | Aislar la zona argumentativa | Medir contra el gold |
| 6 | Generar chunks para el Lab 2 | Unidades de extracción |

Los documentos son **ficticios**, escritos con el formato de la COFECE y del PJF para fines didácticos.

## 0 · Preparación

In [ ]:
!pip install -q pdfplumber pandas

In [ ]:
# Descarga del dataset desde el repositorio público del curso (git viene instalado en Colab).
import os, shutil, zipfile
REPO_URL = "https://github.com/benjov/laboratorio-norma-notebooks"

if not os.path.exists("norma_mini_dataset"):
    !git clone -q --depth 1 "$REPO_URL" repo_curso
    if os.path.exists("repo_curso/norma_mini_dataset"):
        shutil.copytree("repo_curso/norma_mini_dataset", "norma_mini_dataset")
    else:
        print("No se pudo descargar de GitHub. Sube norma_mini_dataset.zip a mano:")
        from google.colab import files
        files.upload()
        zipfile.ZipFile("norma_mini_dataset.zip").extractall(".")
print(sorted(os.listdir("norma_mini_dataset")))

In [ ]:
import re, json, collections
from pathlib import Path
import pdfplumber
import pandas as pd

DATA = Path("norma_mini_dataset")
PDFS = sorted((DATA / "pdf").glob("*.pdf"))
gold = pd.read_csv(DATA / "gold_criterios.csv")
negativos = pd.read_csv(DATA / "negativos_validacion.csv")
[p.name for p in PDFS]

## 1 · Leer el PDF página por página

No basta con el texto: guardamos cada **línea con su posición vertical**. La posición permite distinguir el cuerpo del documento del pie de página y, más adelante, anclar una cita a una caja en la página.

> En producción, Norma+ usa un servicio de OCR y *parsing* porque muchas resoluciones son escaneos. Nuestros PDF ya traen capa de texto, así que basta con `pdfplumber`.

In [ ]:
def leer_pdf(ruta):
    paginas = []
    with pdfplumber.open(ruta) as pdf:
        for n, p in enumerate(pdf.pages, 1):
            paginas.append({
                "pagina": n,
                "alto": p.height,
                "lineas": [{"text": l["text"], "top": l["top"], "bottom": l["bottom"]}
                           for l in p.extract_text_lines()],
            })
    return paginas

docs = {p.name: leer_pdf(p) for p in PDFS}
for nombre, pags in docs.items():
    print(f"{nombre}: {len(pags)} páginas, {sum(len(p['lineas']) for p in pags)} líneas")

In [ ]:
# Las últimas líneas de cada página de R1: ¿qué se repite?
for p in docs["R1_COFECE_PMR_lacteos.pdf"]:
    print(p["pagina"], "→", [l["text"] for l in p["lineas"][-2:]])

## 2 · Detectar y quitar el ruido repetido

Encabezados y pies se repiten en cada página y contaminan el texto: parten oraciones y rompen la búsqueda de citas. La idea es simple:

1. Tomar las líneas de las franjas superior e inferior de cada página.
2. Normalizarlas, reemplazando dígitos por `#` (así, «Página 2» y «Página 3» son la misma firma).
3. Marcar como ruido las firmas que aparecen en la mayoría de las páginas.

### ✏️ Ejercicio 1
Completa `detectar_ruido`. Debe devolver el conjunto de firmas que aparecen en las franjas en al menos `umbral` de las páginas (y nunca menos de 2).

In [ ]:
def firma(texto):
    return re.sub(r"\d+", "#", texto.strip())

def detectar_ruido(paginas, umbral=0.6, franja=0.12):
    conteo = collections.Counter()
    for p in paginas:
        en_franja = {firma(l["text"]) for l in p["lineas"]
                     if l["top"] > (1 - franja) * p["alto"] or l["bottom"] < franja * p["alto"]}
        conteo.update(en_franja)
    minimo = max(2, umbral * len(paginas))
    return {f for f, c in conteo.items() if c >= minimo}

In [ ]:
ruido = {n: detectar_ruido(p) for n, p in docs.items()}
for n, r in ruido.items():
    print(n, "→", r)
assert all(len(r) >= 1 for r in ruido.values()), "Cada documento debería tener al menos una firma de ruido"
print("✅ Ruido detectado")

In [ ]:
def limpiar(paginas, ruido):
    """Devuelve el texto limpio y una lista (offset_inicio, página) para ubicar cualquier offset."""
    texto, mapa = "", []
    for p in paginas:
        mapa.append((len(texto), p["pagina"]))
        lineas = [l["text"] for l in p["lineas"] if firma(l["text"]) not in ruido]
        texto += "\n".join(lineas) + "\n"
    return texto, mapa

def pagina_de(offset, mapa):
    return max(pag for ini, pag in mapa if ini <= offset)

limpios = {n: limpiar(docs[n], ruido[n]) for n in docs}
print(limpios["R1_COFECE_PMR_lacteos.pdf"][0][:600])

## 3 · ¿Qué tipo de procedimiento es? Un router con perfiles

Una resolución de la COFECE y una sentencia de amparo no se leen igual: cambian los nombres de las secciones y dónde está el razonamiento. Norma+ resuelve esto con **un solo pipeline y varios perfiles**. Un *router* decide qué perfil aplicar.

Cada perfil define:
- `señales`: patrones que suelen aparecer en el encabezado del documento.
- `secciones`: los nombres de las secciones que esperamos encontrar, en orden.
- `argumentativa`: la sección donde se razona.

### ✏️ Ejercicio 2
Completa `clasificar`: cuenta cuántas señales de cada perfil aparecen en los primeros 800 caracteres y devuelve el perfil con más coincidencias, o `"DESCONOCIDO"` si ninguno coincide.

In [ ]:
PERFILES = {
    "COFECE_INV": {"señales": [r"Agente económico investigado", r"\bDE-[\w-]+"],
                   "secciones": ["RESULTANDO", "CONSIDERANDO", "RESUELVE"],
                   "argumentativa": "CONSIDERANDO"},
    "COFECE_CNT": {"señales": [r"Notificantes", r"\bCNT-[\w-]+"],
                   "secciones": ["RESULTANDO", "CONSIDERANDO", "RESUELVE"],
                   "argumentativa": "CONSIDERANDO"},
    "PJF_JD":     {"señales": [r"Juzgado .{0,40}de Distrito", r"Amparo indirecto"],
                   "secciones": ["RESULTANDOS", "CONSIDERANDOS", "PUNTOS RESOLUTIVOS"],
                   "argumentativa": "CONSIDERANDOS"},
}

In [ ]:
def clasificar(texto, perfiles=PERFILES, ventana=800):
    cabeza = texto[:ventana]
    puntajes = {nombre: sum(bool(re.search(s, cabeza)) for s in perfil["señales"])
                for nombre, perfil in perfiles.items()}
    mejor = max(puntajes, key=puntajes.get)
    return (mejor if puntajes[mejor] > 0 else "DESCONOCIDO"), puntajes

In [ ]:
esperado = {"R1_COFECE_PMR_lacteos.pdf": "COFECE_INV",
            "R2_COFECE_CNT_farmacias.pdf": "COFECE_CNT",
            "R3_PJF_JD_amparo_lacteos.pdf": "PJF_JD"}
procedimiento = {}
for n, (texto, _) in limpios.items():
    procedimiento[n], puntajes = clasificar(texto)
    print(f"{n:32} → {procedimiento[n]:12} {puntajes}")
assert procedimiento == esperado, "Revisa tu router"
print("✅ Router correcto")

> **Para discutir:** R3 obtiene un punto para `COFECE_INV`: su primer párrafo cita el expediente `DE-…` de la resolución que impugna. El router acierta porque gana por mayoría, pero el empate estuvo cerca. ¿Qué pasaría si buscáramos en todo el texto, donde R3 habla de «práctica monopólica» tanto como R1? ¿Qué señal agregarías para hacerlo más robusto?

## 4 · Mapa de secciones y apartados

Con el perfil sabemos qué secciones esperar. Ahora ubicamos cada sección y cada apartado ordinal («PRIMERO.», «SEGUNDO.»…) con **offsets exactos de carácter**. Los offsets son la base de la trazabilidad: cualquier fragmento que extraigamos después podrá ubicarse en su sección y en su página.

### ✏️ Ejercicio 3
Escribe la expresión regular `RE_APARTADO`. Debe capturar, al inicio de una línea, el ordinal (grupo `ord`) y el título hasta el primer punto (grupo `titulo`). Ejemplo: en `"TERCERO. Mercado relevante. Para determinar…"`, `ord="TERCERO"` y `titulo="Mercado relevante"`.

In [ ]:
ORDINALES = "PRIMERO|SEGUNDO|TERCERO|CUARTO|QUINTO|SEXTO|SÉPTIMO|OCTAVO|NOVENO|DÉCIMO"
RE_APARTADO = re.compile(rf"^(?P<ord>{ORDINALES})\.\s+(?P<titulo>[^.\n]+)", re.MULTILINE)

In [ ]:
def mapa_secciones(nombre):
    texto, mapa = limpios[nombre]
    perfil = PERFILES[procedimiento[nombre]]
    # 1) secciones: líneas que coinciden exactamente con los nombres del perfil
    secciones = []
    for sec in perfil["secciones"]:
        m = re.search(rf"^{sec}$", texto, re.MULTILINE)
        if m:
            secciones.append((sec, m.start()))
    limites = secciones + [("FIN", len(texto))]
    filas = []
    for (sec, ini), (_, fin) in zip(limites, limites[1:]):
        aps = list(RE_APARTADO.finditer(texto, ini, fin))
        for a, sig in zip(aps, aps[1:] + [None]):
            a_fin = sig.start() if sig else fin
            filas.append({"documento": nombre, "procedimiento": procedimiento[nombre],
                          "seccion": sec, "apartado": a["ord"], "titulo": a["titulo"].strip()[:60],
                          "inicio": a.start(), "fin": a_fin,
                          "pag_ini": pagina_de(a.start(), mapa), "pag_fin": pagina_de(a_fin - 1, mapa)})
    return pd.DataFrame(filas)

mapa = pd.concat([mapa_secciones(n) for n in limpios], ignore_index=True)
assert len(mapa) > 0 and mapa["titulo"].str.len().gt(0).all(), "Revisa RE_APARTADO"
print("✅", len(mapa), "apartados")
mapa[["documento", "seccion", "apartado", "titulo", "pag_ini", "pag_fin"]]

## 5 · La zona argumentativa, medida contra el gold

La zona argumentativa es la sección donde la autoridad razona: los considerandos. Si la aislamos bien, el LLM del Lab 2 lee menos texto (menos costo) y con menos distractores (más precisión).

¿Cómo sabemos si la aislamos bien? **Midiendo.** Buscamos la cita literal de cada criterio del gold dentro de la zona. Normalizamos los espacios, porque el PDF parte las líneas donde quiere.

In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", s).strip()

def zona(nombre):
    sec = PERFILES[procedimiento[nombre]]["argumentativa"]
    filas = mapa[(mapa.documento == nombre) & (mapa.seccion == sec)]
    texto, _ = limpios[nombre]
    return norm(texto[filas.inicio.min():filas.fin.max()])

zonas = {n: zona(n) for n in limpios}
for n in limpios:
    print(f"{n:32} zona = {len(zonas[n]):5} de {len(norm(limpios[n][0])):5} caracteres")

gold["en_zona"] = [norm(c) in zonas[d] for d, c in zip(gold.documento, gold.cita_literal)]
negativos["en_zona"] = [norm(t) in zonas[d] for d, t in zip(negativos.documento, negativos.texto_literal)]
print(f"\nCriterios del gold dentro de la zona: {gold.en_zona.sum()} de {len(gold)}")
print(f"Negativos dentro de la zona:          {negativos.en_zona.sum()} de {len(negativos)}")
negativos[["id", "tipo", "en_zona"]]

> **Para discutir:** la zona conserva el 100% de los criterios y descarta parte del ruido, pero no todo. Varios negativos (hechos, transcripciones, argumentos de parte) están *dentro* de los considerandos. Ninguna regla de posición los separa: hace falta entender el texto. Ese es el trabajo del extractor y del validador en el Lab 2.

## 6 · Chunks para el Lab 2

La unidad de extracción será **un apartado de la zona argumentativa**. Cada chunk lleva su documento, procedimiento, apartado y páginas, de modo que todo criterio que el LLM extraiga herede esa trazabilidad.

In [ ]:
chunks = []
for _, f in mapa.iterrows():
    if f.seccion != PERFILES[f.procedimiento]["argumentativa"]:
        continue
    texto, _ = limpios[f.documento]
    chunks.append({"id_chunk": f"{f.documento[:2]}-{f.apartado}", "documento": f.documento,
                   "procedimiento": f.procedimiento, "apartado": f.apartado, "titulo": f.titulo,
                   "pag_ini": int(f.pag_ini), "pag_fin": int(f.pag_fin),
                   "texto": norm(texto[f.inicio:f.fin])})

salida = DATA / "salidas"
salida.mkdir(exist_ok=True)
with open(salida / "chunks_lab1.jsonl", "w", encoding="utf-8") as fh:
    for c in chunks:
        fh.write(json.dumps(c, ensure_ascii=False) + "\n")

df_chunks = pd.DataFrame(chunks)
df_chunks["caracteres"] = df_chunks.texto.str.len()
print(f"✅ {len(chunks)} chunks guardados en {salida / 'chunks_lab1.jsonl'}")
df_chunks[["id_chunk", "titulo", "pag_ini", "pag_fin", "caracteres"]]

## Cierre

| Lo que hicimos | Lo que en Norma+ es |
|---|---|
| `pdfplumber` + líneas con posición | OCR y *parsing* con cajas por página |
| Firmas repetidas en franjas | Limpieza de ruido estructural |
| `PERFILES` + `clasificar` | Router por tipo de procedimiento |
| `RE_APARTADO` + offsets | Mapa del documento |
| Medir la zona contra el gold | Evaluación desde la primera etapa |

**Para ciencias sociales:** este mismo flujo sirve para cualquier corpus semiestructurado: actas legislativas, informes de auditoría, quejas ante comisiones de derechos humanos. Las reglas cambian; la disciplina de medir cada etapa, no.

**Siguiente:** en el Lab 2, Claude extrae criterios de estos chunks con un esquema estructurado y verificamos que cada cita exista literalmente en el texto.